# Data Processing Pipeline

Prepares the Crunchbase 2013 snapshot (Kaggle: `justinas/startup-investments`) for use in the TextGrad-optimized VC investment agent.

**Pipeline steps:**
1. Load merged dataset from parquet
2. Verify data integrity (duplicates)
3. Filter to companies with known outcomes (`acquired`, `ipo`, `closed`)
4. Filter to companies founded between 2005 and 2013 (inclusive)
5. Keep all geographies (no country filter in this processing phase)
6. Filter to companies with substantive overview text (>= 300 characters)
7. Create binary target variable
8. Detect leakage risk in narrative fields (`overview`, `short_description`, `description`)
9. Create anonymized text features (`overview_anon`, `short_description_anon`)
12. Inspect missingness and apply non-statistical column pruning
13. Save cleaned dataset and run reproducibility checks

## Processing Assumptions (2013 Snapshot)

- This is a fixed historical snapshot; labels are defined only from status observed by 2013.
- `operating` is excluded because outcome is unresolved at snapshot time.
- All geographies are retained intentionally in this phase.
- No train/test-time statistical transforms are applied here; those are deferred to the training pipeline to avoid leakage.
- Leakage-prone columns (for example outcome-timing fields) are removed before final save.

## Imports and Reproducibility Setup

In [26]:
import pandas as pd 
import numpy as np
import random
import json
import sys
from pathlib import Path
import re
from google import genai # for Gemini API calls
from groq import Groq # for Groq API calls
from anthropic import Anthropic # for Claude API calls
import os

# Find repo root
current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    if (candidate / "src").exists():
        _repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repository root containing src/")

if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.utils.data_splits import get_splits
from src.utils.llm_client import CachedLLMClient

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Load and Merge Dataset

### Source of `companies.csv`

`companies.csv` is created by the `split_objects_by_entity_type.py` script, which separates the raw Crunchbase-style `objects.csv` rows into entity-specific tables. This notebook loads that derived file as the starting point for the processing pipeline.


In [27]:
# Load the data
companies = pd.read_csv("../data/raw/companies.csv", low_memory=False)
financial_orgs = pd.read_csv("../data/raw/financial_orgs.csv", low_memory=False)
rounds = pd.read_csv("../data/raw/funding_rounds.csv", low_memory=False)
investments = pd.read_csv("../data/raw/investments.csv", low_memory=False)

In [28]:
# Join the dataframes on the company id
companies = companies.rename(columns={'id': 'object_id'})

# Drop pre-computed funding columns that conflict with rounds_agg — we recompute
# them from the actual funding_rounds.csv so the values are self-consistent.
companies = companies.drop(columns=['first_funding_at', 'last_funding_at'], errors='ignore')

# --- Aggregate rounds → one row per company ---
rounds_agg = rounds.groupby('object_id').agg(
    num_rounds        = ('id', 'count'),
    total_raised_usd  = ('raised_amount_usd', 'sum'),
    first_funding_at  = ('funded_at', 'min'),
    last_funding_at   = ('funded_at', 'max'),
    round_types       = ('funding_round_type', lambda x: list(x.dropna().unique()))
).reset_index()

# --- Extract round-by-round funding details (date, type, amount) ---
rounds_sorted = rounds[['object_id', 'funded_at', 'funding_round_type', 'raised_amount_usd']].copy()
rounds_sorted = rounds_sorted.sort_values('funded_at')

def build_round_details(group):
    """Convert grouped rounds into list of dicts with date, type, amount."""
    details = []
    for _, row in group.iterrows():
        details.append({
            'date': str(row['funded_at']) if pd.notna(row['funded_at']) else None,
            'type': str(row['funding_round_type']).lower().replace(' ', '_') if pd.notna(row['funding_round_type']) else 'unknown',
            'amount': float(row['raised_amount_usd']) if pd.notna(row['raised_amount_usd']) else 0
        })
    return details

rounds_details = rounds_sorted.groupby('object_id').apply(build_round_details).reset_index(name='funding_round_details')

# --- Aggregate investments → one row per company ---
investments_agg = investments.groupby('funded_object_id').agg(
    num_investors     = ('investor_object_id', 'nunique'),
    num_invest_rounds = ('funding_round_id', 'nunique')
).reset_index().rename(columns={'funded_object_id': 'object_id'})

# --- Now merge cleanly (all 1:1 at this point) ---
df = companies.merge(rounds_agg,      on='object_id', how='left')
df = df.merge(rounds_details,        on='object_id', how='left')
df = df.merge(investments_agg,      on='object_id', how='left')

# --- Verify: should equal len(companies) ---
assert len(df) == len(companies), f"Duplicates detected! {len(df)} vs {len(companies)}"


In [29]:
# Inspect resulting dataframe
print(companies.shape)
print(companies.columns.tolist())
print(companies.dtypes)

companies = companies.copy()
if 'object_id' in companies.columns:
    companies['object_id'] = companies['object_id'].astype('object')
for col in companies.select_dtypes(include=['string']).columns:
    companies[col] = companies[col].astype('object')

# Apply the same type coercions to df (the merged result that includes funding_round_details)
df = df.copy()
if 'object_id' in df.columns:
    df['object_id'] = df['object_id'].astype('object')
for col in df.select_dtypes(include=['string']).columns:
    df[col] = df[col].astype('object')
# Serialize funding_round_details lists as JSON strings for fastparquet compatibility.
# get_round_summary_from_row() in templates.py already parses JSON strings.
if 'funding_round_details' in df.columns:
    df['funding_round_details'] = df['funding_round_details'].apply(
        lambda x: json.dumps(list(x)) if x is not None and not (isinstance(x, float)) else None
    ).astype(object)

# Convert to parquet
df.to_parquet("../data/processed/combined.parquet", index=False, engine="fastparquet")

# Read back from parquet (much faster for subsequent loads)
df = pd.read_parquet("../data/processed/combined.parquet", engine="fastparquet")


(196553, 38)
['object_id', 'entity_type', 'entity_id', 'parent_id', 'name', 'normalized_name', 'permalink', 'category_code', 'status', 'founded_at', 'closed_at', 'domain', 'homepage_url', 'twitter_username', 'logo_url', 'logo_width', 'logo_height', 'short_description', 'description', 'overview', 'tag_list', 'country_code', 'state_code', 'city', 'region', 'first_investment_at', 'last_investment_at', 'investment_rounds', 'invested_companies', 'funding_rounds', 'funding_total_usd', 'first_milestone_at', 'last_milestone_at', 'milestones', 'relationships', 'created_by', 'created_at', 'updated_at']
object_id                  str
entity_type                str
entity_id                int64
parent_id              float64
name                       str
normalized_name            str
permalink                  str
category_code              str
status                     str
founded_at                 str
closed_at                  str
domain                     str
homepage_url               s

In [30]:
# Print number of unique companies and number of rows in the dataframe
print(f"Number of unique companies: {df['object_id'].nunique()}")

Number of unique companies: 196553


## Team Features

Two-stage aggregation pipeline to add team/education signal from `relationships.csv` and `degrees.csv` without causing row explosion.

**Stages:**
1. Build `company_person_edges` - deduplicated `(company_id, person_id)` edges.
2. Build `person_degree_features` - one row per person with degree counts and top-university indicator.
3. Aggregate to `company_team_features` - one row per company.
4. Left-join into `df` (see bottom of notebook, before save).

Temporal filter: only relationships/degrees plausibly known by 2013 are included.

In [31]:
SNAPSHOT_YEAR = 2013
TOP_UNIVERSITY_COUNT = 50


def _resolve_data_path(*candidates: str) -> str:
    for candidate in candidates:
        candidate_path = Path(candidate)
        if candidate_path.exists():
            return str(candidate_path)
    raise FileNotFoundError(f"None of these paths exist: {candidates}")


QS_RANKINGS_PATH = _resolve_data_path(
    "../data/raw/2024 QS World University Rankings.csv",
    "data/raw/2024 QS World University Rankings.csv",
)


def load_top_universities(rankings_path: str, top_n: int = 50) -> set[str]:
    rankings = pd.read_csv(rankings_path, low_memory=False)

    rank_cols = [col for col in rankings.columns if "RANK" in col.upper()]
    rank_col = next((col for col in rank_cols if "2024" in col.upper()), rank_cols[0] if rank_cols else None)
    if rank_col is None:
        raise ValueError(f"No rank column found in {rankings_path}")

    institution_col = next((col for col in rankings.columns if "INSTITUTION" in col.upper()), None)
    if institution_col is None:
        raise ValueError(f"No institution column found in {rankings_path}")

    top_rankings = rankings.copy()
    top_rankings[rank_col] = pd.to_numeric(
        top_rankings[rank_col].astype(str).str.extract(r"(\d+)")[0],
        errors="coerce",
    )
    top_rankings = top_rankings.dropna(subset=[rank_col, institution_col])
    top_rankings = top_rankings.sort_values(rank_col)
    top_rankings = top_rankings[top_rankings[rank_col] <= top_n]

    return {
        str(name).strip().lower()
        for name in top_rankings[institution_col].dropna().tolist()
        if str(name).strip()
    }


TOP_UNIVERSITIES = load_top_universities(QS_RANKINGS_PATH, top_n=TOP_UNIVERSITY_COUNT)
print(f"Loaded {len(TOP_UNIVERSITIES)} top universities from {QS_RANKINGS_PATH}")

# ── Load raw tables ────────────────────────────────────────────────────────────
rels_raw = pd.read_csv(_resolve_data_path("../data/raw/relationships.csv", "data/raw/relationships.csv"), low_memory=False)
degrees_raw = pd.read_csv(_resolve_data_path("../data/raw/degrees.csv", "data/raw/degrees.csv"), low_memory=False)

# ── Stage 1: company-person edges ─────────────────────────────────────────────
# Temporal filter: drop relationships that started after snapshot year
rels_raw['start_at'] = pd.to_datetime(rels_raw['start_at'], errors='coerce')
rels_co = rels_raw[
    rels_raw['relationship_object_id'].str.startswith('c:', na=False) &
    (rels_raw['start_at'].dt.year.fillna(0).le(SNAPSHOT_YEAR) | rels_raw['start_at'].isna())
].copy()
rels_co = rels_co.rename(columns={
    'relationship_object_id': 'company_id',
    'person_object_id': 'person_id',
})
# Deduplicate: one row per (company, person)
company_person_edges = rels_co[['company_id', 'person_id']].drop_duplicates().copy()

print(f"company_person_edges: {len(company_person_edges):,} rows | "
      f"{company_person_edges['company_id'].nunique():,} unique companies | "
      f"{company_person_edges['person_id'].nunique():,} unique people")

# ── Stage 2: person-level degree features ─────────────────────────────────────
# Temporal filter: exclude degrees graduated after snapshot year
degrees_raw['graduated_at'] = pd.to_datetime(degrees_raw['graduated_at'], errors='coerce')
degrees_filt = degrees_raw[
    degrees_raw['graduated_at'].dt.year.fillna(0).le(SNAPSHOT_YEAR) |
    degrees_raw['graduated_at'].isna()
].copy()

degrees_filt['institution_lower'] = degrees_filt['institution'].fillna('').str.lower().str.strip()
degrees_filt['is_top_university'] = degrees_filt['institution_lower'].apply(
    lambda inst: int(any(u in inst for u in TOP_UNIVERSITIES)) if inst else 0
)

person_degree_features = (
    degrees_filt.groupby('object_id', as_index=False).agg(
        degree_count=('id', 'count'),
        top_university_degree_count=('is_top_university', 'sum'),
    )
    .rename(columns={'object_id': 'person_id'})
)
person_degree_features['has_any_degree'] = (person_degree_features['degree_count'] > 0).astype(int)
person_degree_features['has_top_university'] = (person_degree_features['top_university_degree_count'] > 0).astype(int)

print(f"person_degree_features: {len(person_degree_features):,} people with degree records")

# ── Stage 3: join person features onto edges, aggregate to company ────────────
edges_enriched = company_person_edges.merge(person_degree_features, on='person_id', how='left')
for col in ['degree_count', 'top_university_degree_count', 'has_any_degree', 'has_top_university']:
    edges_enriched[col] = edges_enriched[col].fillna(0).astype(int)

company_team_features = edges_enriched.groupby('company_id', as_index=False).agg(
    team_size=('person_id', 'nunique'),
    person_with_degree_count=('has_any_degree', 'sum'),
    any_top_university_person=('has_top_university', 'max'),
    top_university_person_count=('has_top_university', 'sum'),
)
company_team_features['person_with_degree_pct'] = (
    company_team_features['person_with_degree_count']
    / company_team_features['team_size'].replace(0, float('nan'))
).fillna(0.0)

assert company_team_features['company_id'].is_unique, "Duplicate company_id in company_team_features!"
print(f"company_team_features: {len(company_team_features):,} companies (one row per company)")
print(company_team_features[['team_size', 'person_with_degree_count',
                              'any_top_university_person', 'person_with_degree_pct']].describe().round(2))

Loaded 50 top universities from ../data/raw/2024 QS World University Rankings.csv
company_person_edges: 360,228 rows | 130,378 unique companies | 187,931 unique people
person_degree_features: 68,450 people with degree records
company_team_features: 130,378 companies (one row per company)
       team_size  person_with_degree_count  any_top_university_person  \
count  130378.00                 130378.00                  130378.00   
mean        2.76                      1.52                       0.21   
std         8.51                      7.20                       0.40   
min         1.00                      0.00                       0.00   
25%         1.00                      0.00                       0.00   
50%         1.00                      1.00                       0.00   
75%         3.00                      1.00                       0.00   
max      1093.00                    930.00                       1.00   

       person_with_degree_pct  
count               1

In [32]:
# === BUILD LOOKUP DICTIONARIES FOR FOUNDER AND PRODUCT NAMES ===
print("\nLoading raw data files...")
# Use repo root for reliable path resolution
data_raw = _repo_root / "data" / "raw"

people = pd.read_csv(data_raw / "people.csv", low_memory=False)
print(f"  ✓ Loaded {len(people)} people")

products = pd.read_csv(data_raw / "products.csv", low_memory=False)
print(f"  ✓ Loaded {len(products)} products")

relationships = pd.read_csv(data_raw / "relationships.csv", low_memory=False)
print(f"  ✓ Loaded {len(relationships)} relationships")

# Get companies in our dataset
df_object_ids = set(df['object_id'].unique())
print(f"  ✓ Found {len(df_object_ids)} companies in dataset")

# Build founder/people mapping
print("\nBuilding founder mapping...")
all_founders = (
    relationships[relationships['relationship_object_id'].isin(df_object_ids)]
    .merge(people[['object_id', 'first_name', 'last_name']], 
           left_on='person_object_id', right_on='object_id', how='left')
)

founder_map = {}
for company_id, group in all_founders.groupby('relationship_object_id'):
    names = []
    for _, row in group.iterrows():
        if pd.notna(row['first_name']) and pd.notna(row['last_name']):
            full_name = f"{row['first_name']} {row['last_name']}"
            if full_name not in names:
                names.append(full_name)
        elif pd.notna(row['first_name']):
            if row['first_name'] not in names:
                names.append(row['first_name'])
    if names:
        founder_map[company_id] = names

print(f"  ✓ Built founder map for {len(founder_map)} companies")

# Build product mapping
print("Building product mapping...")
product_map = {}
for company_id, group in products[products['parent_id'].isin(df_object_ids)].groupby('parent_id'):
    product_names = []
    for name in group['name'].dropna().unique():
        if name and len(name) > 1:  # Skip empty or single-char names
            product_names.append(name)
    if product_names:
        product_map[company_id] = product_names

print(f"  ✓ Built product map for {len(product_map)} companies")

print("\nLookup dictionaries ready for use in anonymization.")


Loading raw data files...


  ✓ Loaded 226709 people
  ✓ Loaded 27738 products
  ✓ Loaded 402878 relationships
  ✓ Found 196553 companies in dataset

Building founder mapping...
  ✓ Built founder map for 129329 companies
Building product mapping...
  ✓ Built product map for 11756 companies

Lookup dictionaries ready for use in anonymization.


### Examine Duplicates

In [33]:
# Find rows where id is duplicated
duplicate_ids = df[df.duplicated(subset=["object_id"], keep=False)]["object_id"].unique()
duplicate_names = companies[companies.duplicated(subset=['name'], keep=False)]['name'].nunique()
print(f"Number of duplicate ids: {len(duplicate_ids)}")
print(f"Example duplicate ids: {duplicate_ids[:5]}")
print(f"Number of duplicate names: {duplicate_names}")

# Find differences between rows with the same id
for dup_id in duplicate_ids[:5]:  # Check first 5 duplicate ids
    dup_rows = df[df["id"] == dup_id]
    print(f"\nDuplicate rows for id {dup_id}:")
    print(dup_rows)

Number of duplicate ids: 0
Example duplicate ids: []
Number of duplicate names: 46


In [34]:
# Remove duplicate name rows
df = df.drop_duplicates(subset=['name'], keep='first').copy() 

In [35]:
len(df)

196506

## Target Variable: Subsequent Funding Within 1 Year

The target label captures whether a company received **subsequent funding within 365 days of its first funding date**, using `funding_rounds.csv`. This is a funding-behaviour signal observable at the time of the initial investment decision.

| Target | Label | Definition |
|---|---|---|
| Subsequent funding received | 1 | Any subsequent round within 365 days of `first_funding_at` |
| No subsequent funding | 0 | Single round only, or no funding at all |

**Snapshot boundary**: Companies first funded in 2013 are excluded — their 1-year follow-on window extends beyond the 2013 Crunchbase snapshot, making their label unreliable.


In [36]:
# Status filter removed — target is now funding-behaviour based (follow-on within 1 year),
# not exit-outcome based. All companies with a valid overview are included.
print(f"Total companies before target computation: {len(df)}")
df["status"].value_counts()


Total companies before target computation: 196506


status
operating    183394
acquired       9394
closed         2584
ipo            1134
Name: count, dtype: int64

In [37]:
# Number of successful vs unsuccessful outcomes
df['status'].value_counts()

status
operating    183394
acquired       9394
closed         2584
ipo            1134
Name: count, dtype: int64

In [38]:
# Restrict to cohorts that were plausibly decision-relevant at the 2013 snapshot.
df['founded_at'] = pd.to_datetime(df['founded_at'], errors='coerce')
df = df[df['founded_at'].dt.year >= 2005]
len(df)

67523

## Compute Subsequent Funding Target

For each company, check whether any funding round occurred within 365 days of the first round.
Companies with no `first_funding_at` (never received funding) receive target = 0.
Companies first funded in 2013 are excluded (1-year window extends past snapshot).


In [39]:
# Compute follow-on funding target from funding_rounds.csv
# rounds is already loaded earlier in the notebook (cell 5)
rounds_ts = rounds[["object_id", "funded_at"]].copy()
rounds_ts["funded_at"] = pd.to_datetime(rounds_ts["funded_at"], errors="coerce")
rounds_ts = rounds_ts.dropna(subset=["funded_at"])

def _has_follow_on(group):
    """Return 1 if any round falls strictly after the first funding date and within 365 days.

    Uses dates > first (strict) to exclude same-date duplicate rounds, which represent
    tranches or data artefacts of the initial funding event, not genuine follow-on rounds.
    """
    dates = group["funded_at"].sort_values().reset_index(drop=True)
    first = dates.iloc[0]
    follow_on = dates[(dates > first) & (dates <= first + pd.Timedelta(days=365))]
    return int(len(follow_on) > 0)

follow_on = (
    rounds_ts.groupby("object_id")
    .apply(_has_follow_on, include_groups=False)
    .reset_index(name="target")
)

df = df.merge(follow_on, on="object_id", how="left")
df["target"] = df["target"].fillna(0).astype(int)

# Drop companies with no first_funding_at — target is undefined without a funding date
n_before = len(df)
df = df[df["first_funding_at"].notna()].copy()
print(f"Rows after dropping missing first_funding_at: {len(df)} (removed {n_before - len(df)})")

# Exclude companies first funded in 2013: 1-year follow-on window extends past the snapshot
first_funded_ts = pd.to_datetime(df["first_funding_at"], errors="coerce")
n_before = len(df)
df = df[first_funded_ts < pd.Timestamp("2013-01-01")].copy()
print(f"Rows after 2013-boundary exclusion: {len(df)} (removed {n_before - len(df)})")

print("\nTarget distribution:")
print(df["target"].value_counts())
print(f"Positive rate: {df['target'].mean():.2%}")


Rows after dropping missing first_funding_at: 18321 (removed 49202)
Rows after 2013-boundary exclusion: 14433 (removed 3888)

Target distribution:
target
0    11360
1     3073
Name: count, dtype: int64
Positive rate: 21.29%


In [40]:
# Remove companies whose first funding round is Series A or beyond
import json

LATE_STAGE_ROUND_TYPES = {"series-a", "series-b", "series-c+", "private-equity", "post-ipo"}

def get_first_round_type(rd):
    if not rd or (isinstance(rd, float) and pd.isna(rd)):
        return None
    if isinstance(rd, str):
        try: rd = json.loads(rd)
        except: return None
    if not isinstance(rd, list) or len(rd) == 0:
        return None
    return rd[0].get("type", None)

first_round_types = df["funding_round_details"].apply(get_first_round_type)
before = len(df)
df = df[~first_round_types.isin(LATE_STAGE_ROUND_TYPES)].copy()
print(f"Early-stage filter: removed {before - len(df)} companies (first round Series A or beyond). Remaining: {len(df)}")


Early-stage filter: removed 4116 companies (first round Series A or beyond). Remaining: 10317


In [41]:
# ── Stage 4: left-join company_team_features into df (one-to-one on object_id) ─
rows_before = len(df)
df = df.merge(
    company_team_features.rename(columns={'company_id': 'object_id'}),
    on='object_id',
    how='left',
)
assert len(df) == rows_before, f"Row count changed after team join: {rows_before} → {len(df)}"
assert df['object_id'].is_unique, "Duplicate object_id after team join!"

team_cols = [
    'team_size', 'person_with_degree_count',
    'any_top_university_person', 'top_university_person_count', 'person_with_degree_pct',
]
for col in team_cols:
    df[col] = df[col].fillna(0)
df['team_size'] = df['team_size'].astype(int)

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print("\nTeam feature means by outcome:")
print(df.groupby('target')[team_cols].mean().round(3).to_string())
print("\nSpot-check (top 5 by team_size):")
print(df[['name', 'target', 'team_size', 'any_top_university_person',
          'person_with_degree_pct']].sort_values('team_size', ascending=False).head(5).to_string(index=False))

Rows: 10,317 | Columns: 52

Team feature means by outcome:
        team_size  person_with_degree_count  any_top_university_person  top_university_person_count  person_with_degree_pct
target                                                                                                                     
0           3.220                     1.233                      0.184                        0.342                   0.272
1           5.668                     2.789                      0.367                        0.864                   0.403

Spot-check (top 5 by team_size):
       name  target  team_size  any_top_university_person  person_with_degree_pct
      Zynga       1        107                        1.0                0.710280
    Groupon       1         96                        1.0                0.802083
  Techstars       1         79                        1.0                0.848101
CrowdFlower       1         72                        1.0                0.319444
 

In [42]:
# Verify funding round types are all early-stage
round_types = df['funding_round_details'].apply(get_first_round_type)
if round_types.isin(LATE_STAGE_ROUND_TYPES).any():
    raise ValueError("Some companies still have late-stage first funding rounds after filtering!")

## Filter on `Overview` Quality

The LLM agent's primary input is the `overview` field — a free-text description of the company. Rows with very short overviews provide too little signal for meaningful text-based analysis and would make an LLM-based approach no better than a simpler ML baseline.

A minimum of **300 characters** is applied as the threshold.

In [43]:
# Filter overview column on character length >= 300
df = df[df['overview'].str.len() >= 300]
len(df)

7644

In [44]:
print(df['target'].value_counts())

target
0    5931
1    1713
Name: count, dtype: int64


## Dataset Anonymization

Columns in `objects` that could contain revealing information drop in training pipeline:
- `name`
- `normalized_name`
- `permalink`
- `homepage_url`
- `twitter_username`
- `logo_url`
- `short_description`
- `description`
- `overview`

Columns to anonymize:
- `overview`
- `short_description`

Columns to exclude from training pipeline:
- `name`
- `normalized_name`
- `permalink`
- `homepage_url`
- `twitter_username`
- `logo_url`
- `description`

### Check Information Leakage

In [45]:
# Print sample short_description and description (only print rows with non-null short_description or description)
sample = df[df['short_description'].notna() & df['description'].notna()][['object_id', 'short_description', 'description']].head(10)
print(sample.to_string(index=False))

object_id                                                                                                                            short_description                              description
 c:101271                                                Eqvilibria is a provider of customizable turnkey loyalty and reward programs for major banks.               Loyalty program for banks 
  c:10252                     Mob.ly is a mobile and online recommendations service providing actionable suggestions from friends and trusted sources. mobile and online recommendation service
 c:102867           WeHostels is a mobile app for young travellers to find and book accommodation, discover events, and connect with other travellers.               Youth travel mobile agency
 c:102898   Tasted Menu is a website and mobile app that allows diners to read and review restaurant dishes and make more informed ordering decisions.          Restaurant dish recommendations
 c:102922     EasyOwn.it is an e-commerc

In [46]:
# Check if description columns contain organisation names (potential data leakage)
# For simplicity, we'll just check if the company name appears in the description fields
def check_leakage(row, col='description'):
    name = str(row['name']).lower()
    desc = str(row[col]).lower() if pd.notna(row[col]) else ''
    return int(name in desc)
df['leakage_flag_description'] = df.apply(check_leakage, axis=1)
leakage_count = df['leakage_flag_description'].sum()
print(f"Rows with potential leakage (company name in description): {leakage_count} / {len(df)} ({100*leakage_count/len(df):.2f}%)")


Rows with potential leakage (company name in description): 53 / 7644 (0.69%)


In [47]:
# Check if description columns contain organisation names (potential data leakage)
# For simplicity, we'll just check if the company name appears in the description fields
df['leakage_flag_short_description'] = df.apply(lambda row: check_leakage(row=row, col='short_description'), axis=1)
leakage_count = df['leakage_flag_short_description'].sum()
print(f"Rows with potential leakage (company name in short description): {leakage_count} / {len(df)} ({100*leakage_count/len(df):.2f}%)")


Rows with potential leakage (company name in short description): 1721 / 7644 (22.51%)


In [48]:
# Check if description columns contain former organisation names (potential data leakage)
def check_previous_name_leakage(row):
    string = "formerly known as"
    desc = str(row['overview']).lower() if pd.notna(row['overview']) else ''
    return int(string in desc)
df['leakage_flag_overview'] = df.apply(check_previous_name_leakage, axis=1)
leakage_count = df['leakage_flag_overview'].sum()
print(f"Rows with potential leakage (former company name in overview): {leakage_count} / {len(df)} ({100*leakage_count/len(df):.2f}%)")


Rows with potential leakage (former company name in overview): 97 / 7644 (1.27%)


In [49]:
# Remove leakage flag columns
leakage_cols = [c for c in df.columns if c.startswith('leakage_flag')]
df = df.drop(columns=leakage_cols)

### Analysis Outcome

`description` will be to cumbersome to correctly anonymize and does not provide enough value --> will be excluded from training pipeline.

`short_description` will be anonymized.

## Anonymization of `overview` and `short_description` Columns

Removes identifiable entities while preserving VC-relevant semantics.

In [50]:
# Create anonymized text using startup-name masking + contact masking.
# Keep original columns untouched and write anonymized text to `[column]_anon`.
EMAIL_RE = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b")
URL_RE = re.compile(r"\b(?:https?://|www\.)\S+\b")
PHONE_RE = re.compile(r"(?:(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)\d{3,4}[\s.-]?\d{3,4})")
DOMAIN_RE = re.compile(r"\b(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}\b")
MD_LINK_RE = re.compile(r"\[([^\]]+)\]\(([^\)]+)\)")
LEGAL_SUFFIX_RE = re.compile(
    r"(?:[\s,]*(?:incorporated|inc|corp|corporation|co|company|llc|ltd|limited|plc|gmbh|sa|ag|bv|technologies|systems|solutions|labs|media|networks|digital|interactive))+[\s,.\-]*$",
    flags=re.IGNORECASE,
 )

# Mask old company names
FORMERLY_KNOWN_AS_RE = re.compile(
    r"\s*,?\s*(?:was\s+)?formerly\s+known\s+as\s+[^,.;\n]+,?",
    flags=re.IGNORECASE,
 )
FORMERLY_RE = re.compile(r"\s*,?\s*formerly(?:\s+|,\s*)[^,.;\n]+,?", flags=re.IGNORECASE)
FORMERLY_PAREN_RE = re.compile(r"\(\s*formerly[^)]*\)", flags=re.IGNORECASE)
PREVIOUSLY_KNOWN_AS_RE = re.compile(
    r"\s*,?\s*(?:was\s+)?previously\s+known\s+as\s+[^,.;\n]+,?",
    flags=re.IGNORECASE,
)
PREVIOUSLY_RE = re.compile(r"\s*,?\s*previously(?:\s+|,\s*)[^,.;\n]+,?", flags=re.IGNORECASE)
PREVIOUSLY_PAREN_RE = re.compile(r"\(\s*previously[^)]*\)", flags=re.IGNORECASE)

def _strip_markdown_links(text: str) -> str:
    # Keep visible anchor text and drop markdown URL target.
    return MD_LINK_RE.sub(r'\1', text)

# Remove phrases that indicate previous company names to prevent leakage, e.g. "formerly known as X", "X (formerly Y)", etc.
def _remove_previous_name_phrases(text: str) -> str:
    # Remove patterns like "formerly ...", "formerly known as ...", and common variants.
    text = FORMERLY_KNOWN_AS_RE.sub(' ', text)
    text = FORMERLY_PAREN_RE.sub(' ', text)
    text = FORMERLY_RE.sub(' ', text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    return text.strip()

# Build robust startup-name variants for matching in noisy text, e.g. with/without legal suffixes, punctuation replaced by whitespace, etc.
def _org_name_variants(org_name: str):
    """Build robust startup-name variants for matching in noisy text."""
    variants = set()
    base = org_name.strip()
    if not base:
        return variants

    variants.add(base)

    # Remove trailing legal suffixes: "Inc.", "LLC", etc.
    no_suffix = LEGAL_SUFFIX_RE.sub('', base).strip(' ,.-')
    if no_suffix:
        variants.add(no_suffix)

    # If name is a bare domain, also add leftmost label as potential brand token.
    if re.fullmatch(r"(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}", base):
        label = base.split('.')[0].strip()
        if label:
            variants.add(label)

    # If the name includes punctuation, also try a whitespace-normalized variant.
    normalized = re.sub(r"[^A-Za-z0-9]+", " ", no_suffix if no_suffix else base).strip()
    if normalized:
        variants.add(normalized)
    
    # Also try without spaces (for names like "BlossomandTwigs" that appear as "Blossom and Twigs")
    no_spaces = re.sub(r"\s+", "", base).strip()
    if no_spaces and len(no_spaces) >= 3:
        variants.add(no_spaces)
    
    # Add individual words if name has multiple words (catch partial mentions)
    words = re.findall(r"\b[A-Za-z]+\b", base)
    for word in words:
        if len(word) >= 4:  # Only add words with 4+ chars to avoid false positives
            variants.add(word)

    # Keep meaningful variants only.
    return {v for v in variants if len(v) >= 3}

def _product_name_variants(product_name: str):
    """Generate variants of product names for robust matching."""
    variants = set()
    base = product_name.strip()
    if not base:
        return variants
    
    variants.add(base)
    
    # Remove trailing suffixes (tool, platform, service, etc.)
    no_suffix = re.sub(
        r"\s*(?:tool|platform|service|system|software|app|plugin|module|suite|engine|manager|client|server)[\s,.\-]*$",
        "",
        base,
        flags=re.IGNORECASE
    ).strip()
    if no_suffix and no_suffix != base:
        variants.add(no_suffix)
    
    # Whitespace-normalized variant (e.g., "Spam Guard" -> "spamguard")
    normalized = re.sub(r"[^A-Za-z0-9]+", "", base).strip()
    if normalized and len(normalized) >= 3:
        variants.add(normalized)
    
    # Add space-separated variant
    spaced = re.sub(r"[^A-Za-z0-9]+", " ", base).strip()
    if spaced and len(spaced) >= 3:
        variants.add(spaced)
    
    # Add individual significant words (4+ chars)
    words = re.findall(r"\b[A-Za-z]+\b", base)
    for word in words:
        if len(word) >= 4:
            variants.add(word)
    
    # Acronym: first letter of each word
    if len(words) > 1:
        acronym = "".join(w[0] for w in words)
        if len(acronym) >= 2:
            variants.add(acronym)
    
    return {v for v in variants if len(v) >= 2}

# Mask startup name variants in text with [ORG] placeholder
def _mask_startup_name(text: str, org_name) -> str:
    if not isinstance(org_name, str):
        return text

    variants = sorted(_org_name_variants(org_name), key=len, reverse=True)
    if not variants:
        return text

    masked = text
    for variant in variants:
        # Token-safe boundaries reduce accidental partial-word replacements.
        pattern = re.compile(
            rf"(?<![A-Za-z0-9]){re.escape(variant)}(?![A-Za-z0-9])",
            flags=re.IGNORECASE,
        )
        masked = pattern.sub('[ORG]', masked)
    return masked

def _mask_contacts(text: str) -> str:
    # Keep startup-name masking priority by running this after _mask_startup_name.
    masked = EMAIL_RE.sub('[EMAIL]', text)
    masked = URL_RE.sub('[URL]', masked)
    masked = PHONE_RE.sub('[PHONE]', masked)
    masked = DOMAIN_RE.sub('[URL]', masked)
    return masked

def _mask_people_names(text: str, people_names: list) -> str:
    """Mask people names (founders, employees) with [PERSON] placeholder."""
    if not people_names or not isinstance(text, str):
        return text
    
    # Sort by length descending to match longer names first
    for name in sorted(people_names, key=len, reverse=True):
        if not name or len(name) < 2:
            continue
        
        # Use word boundaries to avoid partial matches
        pattern = re.compile(
            rf"(?<![A-Za-z]){re.escape(name)}(?![A-Za-z])",
            flags=re.IGNORECASE,
        )
        text = pattern.sub('[PERSON]', text)
    
    return text

def _mask_products(text: str, product_names: list) -> str:
    """Mask product/service names with [PRODUCT] placeholder using variant matching."""
    if not product_names or not isinstance(text, str):
        return text
    
    # Build all variants for all products
    all_variants = []
    for product in product_names:
        if product and len(product) >= 2:
            variants = _product_name_variants(product)
            # Store (variant, length) tuples for sorting
            for variant in variants:
                all_variants.append((variant, len(variant)))
    
    # Sort by length descending to match longer patterns first
    all_variants.sort(key=lambda x: x[1], reverse=True)
    
    masked = text
    for variant, _ in all_variants:
        pattern = re.compile(
            rf"(?<![A-Za-z0-9]){re.escape(variant)}(?![A-Za-z0-9])",
            flags=re.IGNORECASE,
        )
        masked = pattern.sub('[PRODUCT]', masked)
    
    return masked

def anonymize_text(text, org_name=None, company_id=None):
    """
    Anonymize text by masking:
    - Company names with [ORG]
    - Founder/employee names with [PERSON] (if company_id provided)
    - Product names with [PRODUCT] (if company_id provided)
    - Contact info (emails, phones, etc.)
    """
    if not isinstance(text, str):
        return text
    
    text = text.strip()
    if not text:
        return text

    text = _strip_markdown_links(text)
    text = _remove_previous_name_phrases(text)
    text = _mask_startup_name(text, org_name)
    
    # Mask people names (if company_id and founder data available)
    if company_id and 'founder_map' in globals() and company_id in founder_map:
        text = _mask_people_names(text, founder_map[company_id])
    
    # Mask products (if company_id and product data available)
    if company_id and 'product_map' in globals() and company_id in product_map:
        text = _mask_products(text, product_map[company_id])
    
    text = _mask_contacts(text)
    return text

# Initial anonymization (Phase 1: company names only)
# This will be expanded after building founder/product maps
name_col = 'object_name' if 'object_name' in df.columns else ('name' if 'name' in df.columns else None)
text_cols = ['overview', 'short_description']

if name_col is None:
    print("Warning: neither 'object_name' nor 'name' exists; masking cannot target startup names.")
    for col in text_cols:
        if col in df.columns:
            df[f'{col}_anon'] = [anonymize_text(text) for text in df[col]]
else:
    for col in text_cols:
        if col in df.columns:
            df[f'{col}_anon'] = [
                anonymize_text(text, org_name=org_name)
                for text, org_name in zip(df[col], df[name_col])
            ]

print('Phase 1 (company name) anonymization complete.')
print(f"Rows: {len(df)}")
print(f"Org-name source column: {name_col}")
for col in text_cols:
    if f'{col}_anon' in df.columns:
        print(f"Created column: {col}_anon")

sample_cols = [c for c in ['overview', 'overview_anon', 'short_description', 'short_description_anon'] if c in df.columns]
sample = df[sample_cols].dropna().head(3)
for i, row in sample.iterrows():
    print(f"\n--- sample index {i} ---")
    if 'overview' in sample_cols:
        print('ORIGINAL OVERVIEW:')
        print(row['overview'][:400])
    if 'overview_anon' in sample_cols:
        print('\nANONYMIZED OVERVIEW:')
        print(row['overview_anon'][:400])
    if 'short_description' in sample_cols:
        print('\n--- short description ---')
        print(row['short_description'][:400])
    if 'short_description_anon' in sample_cols:
        print('\n--- short description anonymized ---')
        print(row['short_description_anon'][:400])

Phase 1 (company name) anonymization complete.
Rows: 7644
Org-name source column: name
Created column: overview_anon
Created column: short_description_anon

--- sample index 14 ---
ORIGINAL OVERVIEW:
Eqvilibria is a leading provider of turnkey loyalty and reward programmes for bank clients. Eqvilibria helps banks and other card issuers to increase the spend on cards through tailored online and offline privilege offerings. Some of the larger clients who use Eqvilibria services are:
 Citi Bank  HSBC  Societe Generale  Banca Intesa  Swedbank  Raiffeisen Bank  MasterCard
http://eqvilibria.r

ANONYMIZED OVERVIEW:
[ORG] is a leading provider of turnkey loyalty and reward programmes for bank clients. [ORG] helps banks and other card issuers to increase the spend on cards through tailored online and offline privilege offerings. Some of the larger clients who use [ORG] services are:  Citi Bank  HSBC  Societe Generale  Banca Intesa  Swedbank  Raiffeisen Bank  MasterCard [URL]

--- 

In [51]:
# === ANONYMIZE FOUNDER AND PRODUCT NAMES ===
id_col = 'object_id'
name_col = 'object_name' if 'object_name' in df.columns else ('name' if 'name' in df.columns else None)
text_cols = ['overview', 'short_description']

print(f"\nAnonymizing {len(df)} rows with ENHANCED variant matching...")
print("  - Product name variants (spacing, suffixes, acronyms)")
print("  - Company name variants (spacing, punctuation, partial words)")
print("  - Founder/employee names")
print("  - Contact information\n")

# Anonymize with extended function that includes founder/product masking
anonymized_count = 0
for col in text_cols:
    if col not in df.columns:
        print(f"  - Skipping {col} (not in dataframe)")
        continue
    
    print(f"  Processing {col}...")
    df[f'{col}_anon'] = df.apply(
        lambda row: anonymize_text(
            row[col] if pd.notna(row[col]) else "",
            org_name=row[name_col] if pd.notna(row[name_col]) else None,
            company_id=row[id_col]
        ),
        axis=1
    )
    anonymized_count += 1
    print(f"    ✓ Anonymized {col} with founder/product masking")

print(f"\n  ✓ Anonymized {anonymized_count} columns (Phase 1 + Phase 2)")


Anonymizing 7644 rows with ENHANCED variant matching...
  - Product name variants (spacing, suffixes, acronyms)
  - Company name variants (spacing, punctuation, partial words)
  - Founder/employee names
  - Contact information

  Processing overview...
    ✓ Anonymized overview with founder/product masking
  Processing short_description...
    ✓ Anonymized short_description with founder/product masking

  ✓ Anonymized 2 columns (Phase 1 + Phase 2)


## Missing Value Analysis

In [52]:
# Print full missing-values summary 
missing_summary = df.isnull().sum().sort_values(ascending=False)
print(missing_summary.to_string())

parent_id                      7644
first_investment_at            7610
last_investment_at             7610
closed_at                      6949
short_description              5711
num_invest_rounds              3123
num_investors                  3123
state_code                     2964
first_milestone_at             2434
last_milestone_at              2434
tag_list                       2363
created_by                     2061
description                    1823
twitter_username               1697
city                            615
country_code                    428
logo_url                        322
domain                           87
homepage_url                     87
category_code                    68
entity_id                         0
object_id                         0
overview                          0
logo_height                       0
logo_width                        0
status                            0
permalink                         0
founded_at                  

## Save Final Processed Dataset

In [53]:
output_path = _repo_root / "data" / "processed" / "companies_clean.parquet"
df.to_parquet(output_path, index=False)

print(f"Saved {len(df)} rows, {df.shape[1]} columns")
print(f"Path: {output_path}")

required_cols = ['object_id', 'status', 'target', 'overview', 'overview_anon']
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns in final dataset: {missing_required}")

print("\nFinal class balance:")
print(df['target'].value_counts(dropna=False).to_string())
print(f"Success rate: {df['target'].mean():.2%}")

print("\nCritical-column missingness:")
critical_cols = ['overview', 'overview_anon', 'category_code', 'country_code', 'city', 'founded_at']
for col in critical_cols:
    if col in df.columns:
        miss = int(df[col].isna().sum())
        miss_pct = 100 * miss / len(df) if len(df) else 0
        print(f"  {col:14s} {miss:6d} ({miss_pct:5.2f}%)")

print("\nSchema preview:")
for col in df.columns:
    print(f"  {col:25s} {str(df[col].dtype)}")

Saved 7644 rows, 54 columns
Path: /Users/louisewiljander/Documents/Projects/llm-vc-decision-textgrad/llm-vc-decision-textgrad/data/processed/companies_clean.parquet

Final class balance:
target
0    5931
1    1713
Success rate: 22.41%

Critical-column missingness:
  overview            0 ( 0.00%)
  overview_anon       0 ( 0.00%)
  category_code      68 ( 0.89%)
  country_code      428 ( 5.60%)
  city              615 ( 8.05%)
  founded_at          0 ( 0.00%)

Schema preview:
  object_id                 object
  entity_type               object
  entity_id                 int64
  parent_id                 float64
  name                      object
  normalized_name           object
  permalink                 object
  category_code             object
  status                    object
  founded_at                datetime64[us]
  closed_at                 object
  domain                    object
  homepage_url              object
  twitter_username          object
  logo_url            

In [54]:
# Print number of rows in each time pool: Train pool: 2005 <= first_funding_at <= 2008, Val pool: first_funding_at == 2009, Test pool: first_funding_at >= 2010
train_pool = df[(df['first_funding_at'] >= '2005-01-01') & (df['first_funding_at'] <= '2008-12-31')]
val_pool = df[(df['first_funding_at'] >= '2009-01-01') & (df['first_funding_at'] <= '2009-12-31')]  
test_pool = df[df['first_funding_at'] >= '2010-01-01']
print(f"Train pool: {len(train_pool)} rows")
print(f"Val pool: {len(val_pool)} rows")
print(f"Test pool: {len(test_pool)} rows")


Train pool: 1377 rows
Val pool: 795 rows
Test pool: 5458 rows


## Data Split

In [55]:
# Split data into train/val/test
df_train, df_val, df_test = get_splits(random_state=SEED)

for split_name, split_df in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"\n{split_name.upper()} SPLIT:")
    print(f"  Rows: {len(split_df)}")
    print(f"  Success rate: {split_df['target'].mean():.2%}")
    split_path = _repo_root / "data" / "processed" / f"companies_{split_name}.parquet"
    split_df.to_parquet(split_path, index=False)
    print(f"  Saved to: {split_path}")


Dataset splits:
  Train                              480 rows   240+ /  240-  (50.0% positive)
  Val                                200 rows   100+ /  100-  (50.0% positive)
  Test (first_funding>=2010)         300 rows    70+ /  230-  (23.3% positive)
  (Rows discarded from train to enforce balance: 897)

TRAIN SPLIT:
  Rows: 480
  Success rate: 50.00%
  Saved to: /Users/louisewiljander/Documents/Projects/llm-vc-decision-textgrad/llm-vc-decision-textgrad/data/processed/companies_train.parquet

VAL SPLIT:
  Rows: 200
  Success rate: 50.00%
  Saved to: /Users/louisewiljander/Documents/Projects/llm-vc-decision-textgrad/llm-vc-decision-textgrad/data/processed/companies_val.parquet

TEST SPLIT:
  Rows: 300
  Success rate: 23.33%
  Saved to: /Users/louisewiljander/Documents/Projects/llm-vc-decision-textgrad/llm-vc-decision-textgrad/data/processed/companies_test.parquet


In [56]:
# Inspect samples from train, val and test sets to verify temporal split
for split_name, split_df in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"\n{split_name.upper()} SPLIT SAMPLE:")
    sample = split_df[['object_id', 'founded_at', 'first_funding_at', 'target']].sort_values('first_funding_at').head(5)
    print(sample.to_string(index=False))


TRAIN SPLIT SAMPLE:
object_id founded_at first_funding_at  target
   c:8689 2005-01-01       2005-01-01       0
  c:27952 2005-01-01       2005-01-01       0
 c:147853 2011-11-18       2005-01-01       0
  c:27777 2005-01-01       2005-01-01       0
   c:3060 2005-03-01       2005-01-01       0

VAL SPLIT SAMPLE:
object_id founded_at first_funding_at  target
  c:54675 2008-01-01       2009-01-01       1
  c:59169 2009-04-27       2009-01-01       0
  c:10817 2008-04-01       2009-01-01       1
  c:58362 2009-04-01       2009-01-01       0
  c:19275 2012-06-11       2009-01-01       0

TEST SPLIT SAMPLE:
object_id founded_at first_funding_at  target
 c:147868 2010-01-23       2010-01-01       0
  c:71107 2010-12-01       2010-01-01       0
  c:64598 2005-12-19       2010-01-01       0
  c:59993 2007-12-01       2010-01-01       0
  c:66621 2010-01-01       2010-01-01       0


## Contamination Probe

Checks whether anonymized test-set descriptions still leak enough identifying signal for an LLM to recover the company name.

**Approach:** an LLM is shown only the anonymized `overview_anon` and `short_description_anon` fields and asked to identify the company. Any row where the model returns a confidence ≥ 70 and names the correct company is flagged for exclusion from the test split.

**Why this matters:** the anonymization pipeline removes the company name, founder names, product names, and contact details, but distinctive product descriptions, funding milestones, or unusual business models can still be fingerprints for well-known startups. The probe catches these residual cases before they bias evaluation results.

**Models used:** Groq (`qwen/qwen3.6-27b`), Gemini (`gemini-2.0-flash-lite`), or Claude Haiku — all with temperature 0 to maximize reproducibility.

In [59]:
# Contamination probe on the eventual test set
#
# This probe only sees anonymized text fields and writes a CSV of object_ids
# that should be excluded from the final test split if the company can be
# identified with reasonable confidence.
CONTAMINATION_REVIEW_PATH = _repo_root / "data" / "processed" / "test_contamination_exclusions.csv"
CONTAMINATION_CONFIDENCE_THRESHOLD = 70

CONTAMINATION_PROBE_SYSTEM_PROMPT = """You are auditing anonymized startup descriptions contamination.
Use only the anonymized description. Do not use the original company name or any external knowledge.

Respond with valid JSON only and exactly these keys:
{
  "identified_company": string or null,
  "confidence": integer 0-100,
  "exclude_from_test": boolean,
  "reasoning": string
}

Set exclude_from_test to true only if you can identify the company with reasonable confidence.
"""


def build_contamination_probe_message(row: pd.Series) -> str:
    """Build the message content for the contamination probe."""
    parts = []
    if pd.notna(row.get('overview_anon')):
        parts.append(f"Overview: {row['overview_anon']}")
    if pd.notna(row.get('short_description_anon')):
        parts.append(f"Short Description: {row['short_description_anon']}")
    return "\n\n".join(parts)


def _extract_json_from_response(response_text: str) -> dict:
    """Extract JSON object from model response, handling extended reasoning and various wrapping formats."""
    # Remove <think>...</think> tags if present (reasoning output from Groq with extended thinking)
    text = re.sub(r"<think>.*?</think>\s*", "", response_text, flags=re.DOTALL)
    
    # First try to find a JSON code block
    code_block_match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if code_block_match:
        try:
            return json.loads(code_block_match.group(1))
        except json.JSONDecodeError:
            pass

    # If no code block, try to find the first JSON object in the text
    # Use a more robust approach: find first '{' and match to corresponding '}'
    start_idx = text.find('{')
    if start_idx != -1:
        # Try to find the matching closing brace by counting braces
        brace_count = 0
        in_string = False
        escape_next = False
        
        for i in range(start_idx, len(text)):
            char = text[i]
            
            if escape_next:
                escape_next = False
                continue
            
            if char == '\\':
                escape_next = True
                continue
            
            if char == '"' and not escape_next:
                in_string = not in_string
                continue
            
            if not in_string:
                if char == '{':
                    brace_count += 1
                elif char == '}':
                    brace_count -= 1
                    if brace_count == 0:
                        # Found the matching closing brace
                        json_str = text[start_idx:i+1]
                        try:
                            return json.loads(json_str)
                        except json.JSONDecodeError:
                            pass
    
    raise ValueError("No JSON object found in response")

In [ ]:
def run_contamination_probe(
    test_df: pd.DataFrame,
    llm_client,
    provider: str = "gemini",
    confidence_threshold: int = CONTAMINATION_CONFIDENCE_THRESHOLD,
    output_path: Path = CONTAMINATION_REVIEW_PATH,
    max_rows: int = None,
) -> pd.DataFrame:
    """Run the contamination probe using LLM API and persist the review results.
    
    Args:
        test_df: DataFrame with test set
        llm_client: API client (Groq, Gemini, or Claude)
        provider: "gemini", "groq", or "claude"
        confidence_threshold: Confidence level to flag for exclusion
        output_path: Path to save results CSV
        max_rows: Limit number of rows to process (for testing)
    
    Note: Free tier APIs have rate limits.
    For Gemini: 5 requests/min
    For Groq free tier: 6000 tokens/min
    For Claude: Generally flexible, no strict rate limits on free tier
    """
    import time
    
    probe_rows = []
    
    # Limit rows for testing if specified
    df_to_process = test_df.head(max_rows) if max_rows else test_df

    for idx, (_, row) in enumerate(df_to_process.iterrows()):
        # Rate limiting
        if idx > 0:
            if provider == "gemini":
                time.sleep(13)  # 13 seconds between Gemini requests (5 req/min)
            else:
                time.sleep(1)   # Groq and Claude are more flexible, minimal delay
        
        message = build_contamination_probe_message(row)
        
        try:
            full_prompt = f"{CONTAMINATION_PROBE_SYSTEM_PROMPT}\n\n{message}"
            
            if provider == "gemini":
                # Gemini API call
                response = llm_client.models.generate_content(
                    model="gemini-2.0-flash-lite",
                    contents=full_prompt,
                    config={
                        "temperature": 0.0,
                        "max_output_tokens": 512,
                    }
                )
                response_text = response.text
                
            elif provider == "groq":
                # Groq API call — uses llama-3.3-70b-versatile (no thinking mode)
                # Note: qwen3.6-27b is reserved for TextGrad; the contamination probe
                # is a one-time preprocessing step and does not need the same model.
                completion = llm_client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[
                        {
                            "role": "system",
                            "content": CONTAMINATION_PROBE_SYSTEM_PROMPT
                        },
                        {
                            "role": "user",
                            "content": message
                        }
                    ],
                    temperature=0.0,
                    max_completion_tokens=512,
                    stream=False,
                )
                response_text = completion.choices[0].message.content
                
            elif provider == "claude":
                # Claude API call
                response = llm_client.messages.create(
                    model="claude-haiku-4-5-20251001",
                    max_tokens=512,
                    temperature=0.0,
                    messages=[
                        {
                            "role": "user",
                            "content": full_prompt
                        }
                    ]
                )
                response_text = response.content[0].text
            else:
                raise ValueError(f"Unknown provider: {provider}. Must be 'gemini', 'groq', or 'claude'.")

            # Parse JSON response
            try:
                parsed = _extract_json_from_response(response_text)
            except (json.JSONDecodeError, ValueError) as e:
                print(f"Warning: Failed to parse JSON for row {idx} (object_id={row.get('object_id')}): {e}")
                print(f"  Response text length: {len(response_text)}, first 300 chars:\n{response_text[:300]}")
                parsed = {}

            # Extract fields with safe defaults
            confidence = parsed.get("confidence", 0)
            if not isinstance(confidence, int):
                try:
                    confidence = int(confidence) if confidence else 0
                except (ValueError, TypeError):
                    confidence = 0
            confidence = max(0, min(100, confidence))  # Clamp to 0-100 range

            identified_company = parsed.get("identified_company")
            
            # Use parsed exclude_from_test if present, else fall back to threshold
            exclude_from_test = parsed.get("exclude_from_test")
            if exclude_from_test is None:
                exclude_from_test = confidence >= confidence_threshold
            else:
                exclude_from_test = bool(exclude_from_test)

            probe_rows.append(
                {
                    "object_id": row.get("object_id"),
                    "name": row.get("name"),
                    "identified_company": identified_company,
                    "confidence": confidence,
                    "exclude_from_test": exclude_from_test,
                    "reasoning": parsed.get("reasoning"),
                }
            )
        
        except Exception as e:
            print(f"Error processing row {idx} (object_id={row.get('object_id')}): {e}")
            # Add a default row indicating we couldn't process this
            probe_rows.append(
                {
                    "object_id": row.get("object_id"),
                    "name": row.get("name"),
                    "identified_company": None,
                    "confidence": 0,
                    "exclude_from_test": False,
                    "reasoning": f"Error: {str(e)}",
                }
            )

    probe_df = pd.DataFrame(probe_rows)
    probe_df.to_csv(output_path, index=False)

    excluded_count = int(probe_df["exclude_from_test"].sum()) if len(probe_df) > 0 else 0
    print(f"\n✓ Saved contamination review to {output_path}")
    print(f"  Provider: {provider}")
    print(f"  Total rows processed: {len(probe_df)}")
    print(f"  Rows flagged for exclusion: {excluded_count} ({100*excluded_count/len(probe_df):.1f}%)")

    return probe_df


SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' on line 60 (1909942124.py, line 70)

### Run Probe

In [ ]:
# Test the contamination probe on a small subset
# Can use Gemini, Groq, or Claude by changing the provider parameter

df_test = pd.read_parquet(_repo_root / "data" / "processed" / "companies_test.parquet")

# # Option 1: Use Gemini
# print("Testing contamination probe with Gemini (20 rows)...")
# print("(with 13-second delays between requests, this will take ~4-5 minutes)")
# client = genai.Client()  # Initialize Gemini client
# probe_results = run_contamination_probe(
#     df_test,
#     client,
#     provider="gemini",
#     max_rows=20
# )

# print("\nContamination probe results summary:")
# print(probe_results["exclude_from_test"].value_counts(dropna=False))
# print("\nFull probe results:")
# print(probe_results.to_string(index=False))

# Option 2: Use Claude Haiku (uncomment to switch providers)
# claude_client = Anthropic()
# print("Testing contamination probe with Claude Haiku (100 rows)...")
# probe_results = run_contamination_probe(
#     df_test,
#     claude_client,
#     provider="claude",
#     max_rows=100
# )

#Option 3: Use Groq (uncomment to switch providers; faster, more lenient rate limits)
groq_client = Groq()
print("Testing contamination probe with Groq (300 rows)...")
probe_results = run_contamination_probe(
    df_test,
    groq_client,
    provider="groq",
    max_rows=300
)

# For full test set (300+ rows):
# probe_results = run_contamination_probe(df_test, client, provider="gemini")


### Review Flagged Rows

Inspect each row where `exclude_from_test=True` to decide whether the company is genuinely identifiable or the model produced a false positive. The raw anonymized text is printed alongside the probe's reasoning to support this judgment call.

In [32]:
# Show flagged rows for manual review
flagged = probe_results[probe_results["exclude_from_test"]]
print(f"\nFlagged {len(flagged)} rows for potential contamination:")
print(flagged[["object_id", "name", "identified_company", "confidence", "reasoning"]].to_string(index=False))

NameError: name 'probe_results' is not defined

In [31]:
# Print overview_anon and short_description_anon for the following list of companies (if they exist in the test set) to help understand why they were flagged for potential contamination:
flagged_object_ids = ['c:29988', 'c:169785', 'c:168546', 'c:190390', 'c:22316', 'c:58387', 'c:59349', 'c:184924', 'c:77507', 'c:37817', 'c:65630', 'c:76658', 'c:164076', 'c:162506', 'c:67880', 'c:58432', 'c:165258']
for obj_id in flagged_object_ids:
    row = df_test[df_test["object_id"] == obj_id]
    if not row.empty:
        print(f"\n--- object_id: {obj_id} | name: {row['name'].values[0]} | target: {row['target'].values[0]} ---")
        overview_anon = row.get("overview_anon", pd.Series([None])).values[0]
        short_desc_anon = row.get("short_description_anon", pd.Series([None])).values[0]
        print(f"ANONYMIZED OVERVIEW:\n{overview_anon}\n")
        print(f"ANONYMIZED SHORT DESCRIPTION:\n{short_desc_anon}\n")
    else:
        print(f"\n--- object_id: {obj_id} not found in test set ---")


--- object_id: c:29988 | name: Newton Peripherals | target: 0 ---
ANONYMIZED OVERVIEW:
[ORG] designs, develops and manufactures the MoGo family of products, including the award-winning MoGo Mouse, a wireless, Bluetooth-enabled, rechargeable mouse specifically designed for mobile computing users. The organization officially unveiled it's flagship product MoGo Mouse at the CES International Las Vegas show in January 2006. The "Mogo Mouse" is protected under U.S. Patent No. 7,233,319 and "MoGo BluetoothÂ® Adapter" technology is patent pending.

ANONYMIZED SHORT DESCRIPTION:



--- object_id: c:169785 | name: Pluribus Networks | target: 0 ---
ANONYMIZED OVERVIEW:
[ORG] is driving the 3rd wave of virtualization into the ethernet hardware market. The company's hardware accelerated, next-generation, virtualized switches and a distributed Netvisor offering a fully programmable network fabric provide true network virtualization for cloud and enterprise data-centers looking for on-demand scale 

In [ ]:
# Show flagged rows  where name and identified_company overlaps for manual review
flagged = probe_results[probe_results["exclude_from_test"]]
count = 0
for i, row in flagged.iterrows():
    count += 1
    if pd.notna(row["identified_company"]) and str(row["identified_company"]) in str(row["name"]):
        print(f"Number of flagged rows: {count}")
        print(row[["object_id", "name", "identified_company", "confidence", "reasoning"]].to_string())

### Remove Contaminated Rows

Drop rows confirmed as identifiable after manual review and save the cleaned test split. Replacement rows (drawn from the same class distribution) are handled inside `get_splits` on next run — see the `Contamination probe exclusions applied` log line above.

In [ ]:
# Remove following companies from test set based on contamination probe results and manual review:
contamination_object_ids = ['c:29988', 'c:169785', 'c:168546', 'c:190390', 'c:58387', 'c:59349', 'c:184924', 'c:77507', 'c:37817', 'c:65630', 'c:76658', 'c:162506', 'c:67880', 'c:58432', 'c:165258']
df_test_cleaned = df_test[~df_test["object_id"].isin(contamination_object_ids)]
print(f"\nCleaned test set rows: {len(df_test_cleaned)} (removed {len(df_test) - len(df_test_cleaned)} contaminated rows)")
cleaned_test_path = _repo_root / "data" / "processed" / "companies_test_cleaned.parquet"
df_test_cleaned.to_parquet(cleaned_test_path, index=False)
print(f"Saved cleaned test set to: {cleaned_test_path}")


Cleaned test set rows: 285 (removed 15 contaminated rows)
Saved cleaned test set to: /Users/louisewiljander/Documents/Projects/llm-vc-decision-textgrad/llm-vc-decision-textgrad/data/processed/companies_test_cleaned.parquet


In [30]:
# Print target distribution in cleaned test set
print("\nCleaned test set target distribution:")
print(df_test_cleaned["target"].value_counts())


Cleaned test set target distribution:
target
0    210
1     75
Name: count, dtype: int64


In [40]:
# === SAMPLE PROFILE INSPECTION ===
# Print one success and one failure from the cleaned test set

import sys
sys.path.insert(0, str(_repo_root))
from src.prompts.templates import format_startup_profile

# Get one success and one failure from the cleaned test set
success_sample = df_test_cleaned[df_test_cleaned["target"] == 1].head(10)
failure_sample = df_test_cleaned[df_test_cleaned["target"] == 0].head(10)

# Print formatted startup profiles for inspection
print("\n=== SUCCESS SAMPLE ===")
for i, row in success_sample.iterrows():
    # Print ID, name, and target for context
    print(f"\n--- object_id: {row['object_id']} | name: {row['name']} | target: {row['target']} ---")
    profile_text = format_startup_profile(row)
    print(profile_text)

print("\n=== FAILURE SAMPLE ===")
for i, row in failure_sample.iterrows():
    # Print ID, name, and target for context
    print(f"\n--- object_id: {row['object_id']} | name: {row['name']} | target: {row['target']} ---")
    profile_text = format_startup_profile(row)
    print(profile_text)



=== SUCCESS SAMPLE ===

--- object_id: c:267247 | name: New Century Hospice | target: 1 ---
DESCRIPTION:
[ORG] is a program of care and support for a person diagnosed with a terminal illness and comfort care is desired. Helping people find meaning and peace is at the core of our mission. At [ORG], our program supports early involvement where a dramatic difference in the quality of life can be experienced. By enabling families and loved ones to participate in the care experience, we are able to make the remaining time more meaningful, dignified and comfortable.
SECTOR: biotech
LOCATION: Dallas, USA
FOUNDED: 2006
INITIAL FUNDING ROUND: Venture (2011): $1,000,000
RELATIONSHIP COUNT: 0
TEAM MEMBERS WITH DEGREE: 0
TOP UNIVERSITY ALUMNI ON TEAM: No

--- object_id: c:69285 | name: HotelTonight | target: 1 ---
DESCRIPTION:
[ORG] is the ultimate way to book a same-day hotel stay via your mobile device. Founded in 2010, [ORG] is the first hotel booking application that is made for mobile from t